In [0]:
import subprocess
import time
from datetime import datetime, timezone

# --- Configuration -----------------------------------------------------------------
REPO_DIR = "/Users/mahadevmaddy15@gmail.com/databricks"   # <-- update: path to your git repo
GIT_REMOTE = "origin"
GIT_BRANCH = "main"
POLL_INTERVAL_SECONDS = 300                     # how often to check for activity
CONTINUOUS = False                              # True = keep polling; False = run once
# ------------------------------------------------------------------------------------


def run_git(*args):
    """Run a git command inside REPO_DIR; return stdout, raise on failure."""
    result = subprocess.run(
        ["git", "-C", REPO_DIR, *args],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{result.stderr.strip()}")
    return result.stdout.strip()


def commit_active_changes():
    """Stage and commit any active (uncommitted) changes. Returns True if committed."""
    if not run_git("status", "--porcelain"):
        return False
    run_git("add", "-A")
    run_git(
        "commit",
        "-m",
        f"Auto-commit by tracker at {datetime.now(timezone.utc).isoformat()}",
    )
    return True


def push_new_commits():
    """Fetch remote state, report commits not yet pushed, and push them."""
    run_git("fetch", GIT_REMOTE)
    last_pushed = run_git("rev-parse", f"{GIT_REMOTE}/{GIT_BRANCH}")
    new_commits = run_git("log", "--oneline", f"{last_pushed}..HEAD")
    if new_commits:
        run_git("push", GIT_REMOTE, GIT_BRANCH)
        print(f"Pushed {len(new_commits.splitlines())} commit(s):\n{new_commits}")
    else:
        print("No new commits to push.")


def track_and_push():
    """Track active commits and push them to the remote repo."""
    print(f"Tracking commits in {REPO_DIR} (branch: {GIT_BRANCH})")
    while True:
        try:
            if commit_active_changes():
                print("Committed active changes.")
            push_new_commits()
        except Exception as exc:
            print(f"Error: {exc}")
        if not CONTINUOUS:
            break
        time.sleep(POLL_INTERVAL_SECONDS)


track_and_push()

# Notes:
# - Update REPO_DIR to your actual repo path (e.g. under /Workspace/Repos/...).
# - Pushing requires git credentials with write access to the remote. In a Databricks
#   Repo you can also use the built-in workspace Git integration (Repos UI / git commands
#   in the notebook's source control panel) instead of a personal access token.
# - Set CONTINUOUS = True to poll every POLL_INTERVAL_SECONDS in a long-running job.
